In [129]:
import os
import urllib.request
import pandas as pd

# 1. Create dataset folder

DATA_FOLDER = "data"
DATA_URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
DATA_FILE = os.path.join(DATA_FOLDER, "sms.tsv")

if not os.path.exists(DATA_FOLDER):
    os.makedirs(DATA_FOLDER)
    print("Created data folder")

if not os.path.exists(DATA_FILE):
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
    print("Dataset downloaded")
else:
    print("Dataset already exists")

# 2. Load dataset

data = pd.read_csv(DATA_FILE, sep="\t", header=None, names=["label", "message"])

# Convert labels to numeric
data["label"] = data["label"].map({"ham": 0, "spam": 1})

# 3. Train/test split

from sklearn.model_selection import train_test_split

X = data["message"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Text vectorization

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 5. Train model

from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_vec, y_train)

# 6. Evaluate model

from sklearn.metrics import accuracy_score
predictions = model.predict(X_test_vec)
print("Model Accuracy:", accuracy_score(y_test, predictions))


Dataset already exists
Model Accuracy: 0.97847533632287


In [131]:
print("Total messages:", len(data))
print("Spam messages:", sum(data["label"]))
print("Ham messages:", len(data) - sum(data["label"]))

Total messages: 5572
Spam messages: 747
Ham messages: 4825


In [145]:
# 7.Test custom email

test_email = ["You have won a FREE prize worth $5000. Call now and claim the reward!"]

test_vec = vectorizer.transform(test_email)
prediction = model.predict(test_vec)

print("\nTest Email:", test_email[0])
print("Prediction:", "Spam" if prediction[0] == 1 else "Ham")


Test Email: You have won a FREE prize worth $5000. Call now and claim the reward!
Prediction: Spam


In [147]:
probs = model.predict_proba(test_vec)

print("Spam probability:", probs[0][1])
print("Ham probability:", probs[0][0])

Spam probability: 0.9528072964015348
Ham probability: 0.04719270359846432
